### Iniciación del cuaderno de experimentación

In [18]:
import os
import sys
from pathlib import Path
import json
import re

# Ruta raíz del proyecto
ruta_raiz = str(Path(os.getcwd()).resolve().parent.parent)
if ruta_raiz not in sys.path:
    sys.path.insert(0, ruta_raiz)

# Habilitar recarga automática de módulos de Python
%load_ext autoreload
%autoreload 2

print(f"✅ Entorno configurado. Ruta raíz: {ruta_raiz}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Entorno configurado. Ruta raíz: C:\Users\inter\OneDrive - Universidad Externado de Colombia\Cuarto Semestre\Seminario Modelos Multiagentes\Talleres\agente_inversiones


### Importación e iniciación de arquitectura

In [19]:
import ollama
from IPython.display import Markdown, display

# Importar el motor principal y modelos
from agente_inversiones.agente import AgenteInversion
from agente_inversiones.modelos import EntradaInversion
from agente_inversiones.prompts_agent import SYSTEM_PROMPT, build_final_answer_prompt

# Inicialización
agente = AgenteInversion()
modelo_llm = "qwen2.5:3b" 

print(f"✅ Arquitectura cargada. Motor LLM configurado con: {modelo_llm}")

✅ Arquitectura cargada. Motor LLM configurado con: qwen2.5:3b


### Planificación

Algunas preguntas de ejemplo, para probar el agente:

- Quiero sacar un crédito de 120,000,000 de pesos. La tasa es del 18.5% efectiva anual y quiero pagarlo a 72 meses. ¿Me puedes dar el resumen?

- ¿Cuánto termino pagando de intereses por un préstamo de 35 millones al 21% efectivo anual a 48 meses?

- ¿Me conviene tomar un crédito de 50 millones al 22% efectivo anual a 60 meses?

In [20]:
# Pregunta usuario
pregunta_usuario = "Quiero sacar un crédito de 120,000,000 de pesos. La tasa es del 18.5% efectiva anual y quiero pagarlo a 72 meses. ¿Me puedes dar el resumen?"

print("🧠 Analizando la intención y extrayendo parámetros...")

prompt_extraccion = f"""
Extrae los parámetros de esta pregunta: '{pregunta_usuario}'
Devuelve ÚNICAMENTE un JSON válido con las siguientes claves y tipos de datos:
{{"monto": float, "tasa": float (en decimal), "tipo_tasa": "efectiva_anual", "periodicidad": "mensual", "plazo_periodos": int, "tipo": "credito"}}
No incluyas texto adicional ni formato markdown en tu respuesta.
"""

resp_extraccion = ollama.chat(
    model=modelo_llm, 
    messages=[{"role": "user", "content": prompt_extraccion}]
)

# Limpieza del texto por si el LLM incluye formato markdown accidental
texto_json = re.sub(r'```json\n|```', '', resp_extraccion['message']['content']).strip()

print("✅ Datos extraídos correctamente:")
print(texto_json)

🧠 Analizando la intención y extrayendo parámetros...


2026-06-08 19:20:56,948 [INFO] httpx :: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


✅ Datos extraídos correctamente:
{"monto": 120000000.0, "tasa": 0.185, "tipo_tasa": "efectiva_anual", "periodicidad": "mensual", "plazo_periodos": 72, "tipo": "credito"}


### Ejecución de la calculadora

In [21]:
print("🧮 Ejecutando simulación financiera estricta...")

try:
    # 1. Convertimos el texto JSON a un diccionario de Python
    datos_extraidos = json.loads(texto_json)
    
    # 2. Validamos los datos usando tu modelo de Pydantic
    entrada = EntradaInversion(**datos_extraidos)
    
    # 3. Ejecutamos el cálculo matemático
    resultado_matematico = agente.simular(entrada)
    
    print("✅ Cálculo completado con éxito. Resultados listos para análisis.")
except json.JSONDecodeError:
    print("❌ Error: El LLM no devolvió un JSON válido. Revisa la celda anterior.")
except Exception as e:
    print(f"❌ Error en la validación o cálculo: {e}")

2026-06-08 19:20:57,115 [INFO] agente :: ▶ INICIO :: Simulación
2026-06-08 19:20:57,116 [INFO] agente :: ▶ INICIO :: Cálculo Crédito
2026-06-08 19:20:57,119 [INFO] agente :: ✓ FIN    :: Cálculo Crédito
2026-06-08 19:20:57,120 [INFO] agente :: CalculadoraCredito.calcular ejecutado en 4.16 ms
2026-06-08 19:20:57,122 [INFO] agente :: AgenteInversion.simular ejecutado en 5.53 ms
2026-06-08 19:20:57,123 [INFO] agente :: ✓ FIN    :: Simulación


🧮 Ejecutando simulación financiera estricta...
✅ Cálculo completado con éxito. Resultados listos para análisis.


### Análisis del experto, revisión del agente

In [ ]:
print("📝 Redactando el informe financiero...\n")

# Construimos el prompt final con constructores profesionales
prompt_final = build_final_answer_prompt(
    user_question=pregunta_usuario, 
    tool_results=resultado_matematico.model_dump_json(indent=2)
)

# Invocamos al LLM con el rol de experto
respuesta_final = ollama.chat(
    model=modelo_llm,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_final}
    ]
)

# Visualizamos la respuesta con formato rico en Jupyter
display(Markdown(respuesta_final['message']['content']))# Fin del código

📝 Redactando el informe financiero...



2026-06-08 19:21:53,700 [INFO] httpx :: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


# Resumen Ejecutivo

El valor futuro del crédito es de aproximadamente $192,664,370.00 COP. La rentabilidad porcentual anual (TEA) para este crédito es del -60.55%. El tipo efectivo anual (TNA) asciende a 18.50%.

# Hallazgos cuantitativos

- **Valor Futuro**: $192,664,370.00 COP
- **Rentabilidad Absoluta y Porcentual**: -60.55%
- **Tipo Efectivo Anual (TEA)**: 18.50%
- **Tasa Periodica Aplicada**: 1.42% mensual

# Interpretación de rentabilidad y riesgo 

Las cifras obtenidas indican un alto costo financiero para este crédito. La TNA del 18.50% es significativa, lo que implica un rendimiento anual real superior al nominal. Sin embargo, la rentabilidad absoluta negativa (-60.55%) indica una situación desfavorable para el solicitante.

Para inversionistas en renta variable, este tipo de rendimientos sería visto con cautela debido a su bajo potencial de ganancias. Para inversores en CDT, es importante mencionar la retención del 7% sobre los interesos cuando superen los topes UVT vigentes, lo que podría afectar el rendimiento real.

# Limitaciones

El modelo asume un tipo de tasa efectiva anual (TEA) de 18.50%, sin considerar posibles cambios futuros en la tasa. Además, el valor futuro del crédito está basado en una retención en la fuente del 7% sobre los interesos superiores a los topes UVT vigentes, lo que no se ha considerado en las cifras proporcionadas.

# Próximos análisis sugeridos

Es recomendable realizar sensibilidades al respecto de los supuestos del modelo, como por ejemplo variaciones en la tasa de interés o cambios en el plazo. También es útil comparar este crédito con otros ofrecimientos para verificar si ofrece una rentabilidad competitiva y considerar escenarios alternativos como cambiar a un plazo más largo o reducir la cuota mensual.

Esto no constituye asesoría personalizada, solo se presentan hallazgos basados en los datos de las herramientas.